In [76]:
# confirm in our virtual environment
import sys
print(sys.executable)

/Users/hussaintaheri/Desktop/sports-win-predictor/venv/bin/python


In [77]:
import os 
print(os.getcwd())

/Users/hussaintaheri/Desktop/sports-win-predictor/notebooks


In [78]:
import pandas as pd
df = pd.read_csv("../data/nba_pbp_2020_2025.csv")
df.head()

,Unnamed: 0.1,Unnamed: 0,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,...,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0,0,22401186,2,PT12M00.00S,1,0,NaN,0,NaN,...,0.0,0.0,0,NaN,Start of 1st Period (1:12 PM EST),period,start,0,0,1
1,1,1,22401186,4,PT12M00.00S,1,1610612737,ATL,1631230,Barlow,...,NaN,NaN,0,h,Jump Ball Barlow vs. Carter Jr.: Tip to Mann,Jump Ball,NaN,1,0,2
2,2,2,22401186,7,PT11M43.00S,1,1610612737,ATL,1629611,Mann,...,NaN,NaN,0,h,MISS Mann 4' Fadeaway Jumper,Missed Shot,Fadeaway Jump Shot,1,2,3
3,3,3,22401186,8,PT11M40.00S,1,1610612753,ORL,1628976,Carter Jr.,...,NaN,NaN,0,v,Carter Jr. REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,4
4,4,4,22401186,9,PT11M23.00S,1,1610612753,ORL,203484,Caldwell-Pope,...,NaN,NaN,0,v,MISS Caldwell-Pope 25' 3PT Jump Shot,Missed Shot,Jump Shot,1,3,5


In [79]:
# view columns to drop which ones we want be using for our model
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'gameId', 'actionNumber', 'clock',
       'period', 'teamId', 'teamTricode', 'personId', 'playerName',
       'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult',
       'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location',
       'description', 'actionType', 'subType', 'videoAvailable', 'shotValue',
       'actionId'],
      dtype='object')

In [80]:
df = df.drop(columns = ['Unnamed: 0.1', 'Unnamed: 0', 'actionNumber', 'teamTricode', 'personId', 'playerName',
       'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult',
       'isFieldGoal', 'description', 'subType', 'videoAvailable', 'shotValue',
       'actionId', 'pointsTotal', 'teamId'], index = 1)

In [81]:
df.isna().count()

gameId        3282178
clock         3282178
period        3282178
scoreHome     3282178
scoreAway     3282178
location      3282178
actionType    3282178
dtype: int64

In [82]:
df.head(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType
0,22401186,PT12M00.00S,1,0.0,0.0,NaN,period
2,22401186,PT11M43.00S,1,NaN,NaN,h,Missed Shot
3,22401186,PT11M40.00S,1,NaN,NaN,v,Rebound
4,22401186,PT11M23.00S,1,NaN,NaN,v,Missed Shot
5,22401186,PT11M19.00S,1,NaN,NaN,h,Rebound
6,22401186,PT11M15.00S,1,2.0,0.0,h,Made Shot
7,22401186,PT10M58.00S,1,NaN,NaN,v,Missed Shot
8,22401186,PT10M55.00S,1,NaN,NaN,v,Rebound
9,22401186,PT10M49.00S,1,2.0,3.0,v,Made Shot
10,22401186,PT10M35.00S,1,NaN,NaN,h,Turnover


In [83]:
# fill missing values for home and away so it carries the values forward and not update whenever a score changes
# use forward fill method
df[["scoreHome","scoreAway"]] = df.groupby(["gameId"])[["scoreHome", "scoreAway"]].ffill()

In [84]:
df.head(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType
0,22401186,PT12M00.00S,1,0.0,0.0,NaN,period
2,22401186,PT11M43.00S,1,0.0,0.0,h,Missed Shot
3,22401186,PT11M40.00S,1,0.0,0.0,v,Rebound
4,22401186,PT11M23.00S,1,0.0,0.0,v,Missed Shot
5,22401186,PT11M19.00S,1,0.0,0.0,h,Rebound
6,22401186,PT11M15.00S,1,2.0,0.0,h,Made Shot
7,22401186,PT10M58.00S,1,2.0,0.0,v,Missed Shot
8,22401186,PT10M55.00S,1,2.0,0.0,v,Rebound
9,22401186,PT10M49.00S,1,2.0,3.0,v,Made Shot
10,22401186,PT10M35.00S,1,2.0,3.0,h,Turnover


In [85]:
# get the last row of every game
temp_df = df.groupby("gameId").last().reset_index()

# create target column for both dataframes comparing to see if home team won or not
temp_df["home_team_won"] = (temp_df["scoreHome"] > temp_df["scoreAway"]).astype(int)

# now merge both dataframes so we know the target value for all rows
df = pd.merge(df, temp_df[["home_team_won", "gameId"]], on = "gameId")

In [86]:
df.head(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType,home_team_won
0,22401186,PT12M00.00S,1,0.0,0.0,NaN,period,1
1,22401186,PT11M43.00S,1,0.0,0.0,h,Missed Shot,1
2,22401186,PT11M40.00S,1,0.0,0.0,v,Rebound,1
3,22401186,PT11M23.00S,1,0.0,0.0,v,Missed Shot,1
4,22401186,PT11M19.00S,1,0.0,0.0,h,Rebound,1
5,22401186,PT11M15.00S,1,2.0,0.0,h,Made Shot,1
6,22401186,PT10M58.00S,1,2.0,0.0,v,Missed Shot,1
7,22401186,PT10M55.00S,1,2.0,0.0,v,Rebound,1
8,22401186,PT10M49.00S,1,2.0,3.0,v,Made Shot,1
9,22401186,PT10M35.00S,1,2.0,3.0,h,Turnover,1


In [87]:
df.tail(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType,home_team_won
3282158,22000496,PT00M06.90S,4,119.0,122.0,h,Rebound,0
3282159,22000496,PT00M06.90S,4,119.0,122.0,v,Foul,0
3282160,22000496,PT00M06.90S,4,120.0,122.0,h,Free Throw,0
3282161,22000496,PT00M06.90S,4,120.0,122.0,v,Substitution,0
3282162,22000496,PT00M06.90S,4,120.0,122.0,h,Free Throw,0
3282163,22000496,PT00M05.80S,4,120.0,122.0,v,Rebound,0
3282164,22000496,PT00M05.80S,4,120.0,122.0,h,Foul,0
3282165,22000496,PT00M05.80S,4,120.0,123.0,v,Free Throw,0
3282166,22000496,PT00M05.80S,4,120.0,123.0,h,Substitution,0
3282167,22000496,PT00M05.80S,4,120.0,124.0,v,Free Throw,0


In [88]:
df.dtypes

gameId             int64
clock             object
period             int64
scoreHome        float64
scoreAway        float64
location          object
actionType        object
home_team_won      int64
dtype: object

In [89]:
# check how much periods in our df
df["period"].value_counts()

period
4    832755
2    830947
3    804796
1    791902
5     19474
6      2076
7       228
Name: count, dtype: int64

In [90]:
df.head()

,gameId,clock,period,scoreHome,scoreAway,location,actionType,home_team_won
0,22401186,PT12M00.00S,1,0.0,0.0,NaN,period,1
1,22401186,PT11M43.00S,1,0.0,0.0,h,Missed Shot,1
2,22401186,PT11M40.00S,1,0.0,0.0,v,Rebound,1
3,22401186,PT11M23.00S,1,0.0,0.0,v,Missed Shot,1
4,22401186,PT11M19.00S,1,0.0,0.0,h,Rebound,1


In [91]:
# created a function to convert clock and period columns to a single column of just seconds
def parse_clock(clock, period):
    clock = clock.split("M")
    clock[0] = clock[0][2:]
    clock[1] = clock[1][:-4]

    clock_seconds_left = int(clock[0]) * 60 + int(clock[1])

    if period < 5:
        seconds_before_period = (period - 1) * 720
        seconds = seconds_before_period + (720 - clock_seconds_left)
    else:
        seconds_before_period = 2880 + (period - 5) * 300
        seconds = seconds_before_period + (300 - clock_seconds_left)

    return seconds

In [92]:
# apply the function to the data frame
df["seconds"] = df.apply(lambda row: parse_clock(row["clock"], row["period"]), axis = 1)
df.head(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType,home_team_won,seconds
0,22401186,PT12M00.00S,1,0.0,0.0,NaN,period,1,0
1,22401186,PT11M43.00S,1,0.0,0.0,h,Missed Shot,1,17
2,22401186,PT11M40.00S,1,0.0,0.0,v,Rebound,1,20
3,22401186,PT11M23.00S,1,0.0,0.0,v,Missed Shot,1,37
4,22401186,PT11M19.00S,1,0.0,0.0,h,Rebound,1,41
5,22401186,PT11M15.00S,1,2.0,0.0,h,Made Shot,1,45
6,22401186,PT10M58.00S,1,2.0,0.0,v,Missed Shot,1,62
7,22401186,PT10M55.00S,1,2.0,0.0,v,Rebound,1,65
8,22401186,PT10M49.00S,1,2.0,3.0,v,Made Shot,1,71
9,22401186,PT10M35.00S,1,2.0,3.0,h,Turnover,1,85


In [93]:
df.tail(20)

,gameId,clock,period,scoreHome,scoreAway,location,actionType,home_team_won,seconds
3282158,22000496,PT00M06.90S,4,119.0,122.0,h,Rebound,0,2874
3282159,22000496,PT00M06.90S,4,119.0,122.0,v,Foul,0,2874
3282160,22000496,PT00M06.90S,4,120.0,122.0,h,Free Throw,0,2874
3282161,22000496,PT00M06.90S,4,120.0,122.0,v,Substitution,0,2874
3282162,22000496,PT00M06.90S,4,120.0,122.0,h,Free Throw,0,2874
3282163,22000496,PT00M05.80S,4,120.0,122.0,v,Rebound,0,2875
3282164,22000496,PT00M05.80S,4,120.0,122.0,h,Foul,0,2875
3282165,22000496,PT00M05.80S,4,120.0,123.0,v,Free Throw,0,2875
3282166,22000496,PT00M05.80S,4,120.0,123.0,h,Substitution,0,2875
3282167,22000496,PT00M05.80S,4,120.0,124.0,v,Free Throw,0,2875


In [94]:
# count the amount of values for seconds
df["seconds"].isna().sum()

np.int64(0)

In [95]:
# now drop the clock column
df = df.drop(["clock"], axis = 1)
df.head()

,gameId,period,scoreHome,scoreAway,location,actionType,home_team_won,seconds
0,22401186,1,0.0,0.0,NaN,period,1,0
1,22401186,1,0.0,0.0,h,Missed Shot,1,17
2,22401186,1,0.0,0.0,v,Rebound,1,20
3,22401186,1,0.0,0.0,v,Missed Shot,1,37
4,22401186,1,0.0,0.0,h,Rebound,1,41


In [96]:
df.dtypes

gameId             int64
period             int64
scoreHome        float64
scoreAway        float64
location          object
actionType        object
home_team_won      int64
seconds            int64
dtype: object

In [97]:
# check the amount of different values in actiontype
df["actionType"].value_counts()

actionType
Rebound           698025
Missed Shot       632259
Made Shot         555059
Substitution      314165
Free Throw        296706
Foul              266823
Turnover          186887
Timeout            73560
period             54470
Instant Replay     12557
Jump Ball          11786
Violation          11652
Ejection             482
Name: count, dtype: int64

In [98]:
# create 13 new columns and a 1 for what actiontype it is and the rest 0s to have our model to be able to know the importance of each type
df = pd.get_dummies(df, columns = ["actionType"], dtype = int)
df.head()

,gameId,period,scoreHome,scoreAway,location,home_team_won,seconds,actionType_Ejection,actionType_Foul,actionType_Free Throw,actionType_Instant Replay,actionType_Jump Ball,actionType_Made Shot,actionType_Missed Shot,actionType_Rebound,actionType_Substitution,actionType_Timeout,actionType_Turnover,actionType_Violation,actionType_period
0,22401186,1,0.0,0.0,NaN,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,22401186,1,0.0,0.0,h,1,17,0,0,0,0,0,0,1,0,0,0,0,0,0
2,22401186,1,0.0,0.0,v,1,20,0,0,0,0,0,0,0,1,0,0,0,0,0
3,22401186,1,0.0,0.0,v,1,37,0,0,0,0,0,0,1,0,0,0,0,0,0
4,22401186,1,0.0,0.0,h,1,41,0,0,0,0,0,0,0,1,0,0,0,0,0


In [99]:
df.dtypes

gameId                         int64
period                         int64
scoreHome                    float64
scoreAway                    float64
location                      object
home_team_won                  int64
seconds                        int64
actionType_Ejection            int64
actionType_Foul                int64
actionType_Free Throw          int64
actionType_Instant Replay      int64
actionType_Jump Ball           int64
actionType_Made Shot           int64
actionType_Missed Shot         int64
actionType_Rebound             int64
actionType_Substitution        int64
actionType_Timeout             int64
actionType_Turnover            int64
actionType_Violation           int64
actionType_period              int64
dtype: object